# IMPORT LIBRARY

In [ ]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import psycopg2

import requests
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import pymysql
pd.set_option('display.max_columns', None)

# CONFIG DB

In [ ]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh = module_from_file("config_dw", "../db_config/config_db_datawarehouse.py")
config_jiwaplus = module_from_file("config_dw", "../db_config/config_db_jiwaplus.py")
config_ksj_link = module_from_file("config_dw", "../db_config/config_db_ksj_link.py")

In [ ]:
def dl_dwh_db_conn(db_name):
    return pymysql.connect(
        host=config_dl_dwh.hostname,
        db=db_name,
        user=config_dl_dwh.username,
        password=config_dl_dwh.password,
        port=3306
    )

In [ ]:
def jiwa_db_conn():
    return psycopg2.connect(
        host=config_jiwaplus.hostname,
        database=config_jiwaplus.database,
        user=config_jiwaplus.username,
        password=config_jiwaplus.password,
        port='5432'
    )

In [ ]:
def ksj_link_db_conn(db_name):
    return psycopg2.connect(
        host=config_ksj_link.hostname,
        database=db_name,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        port='5432'
    )

# FUNCTION

In [ ]:
def get_jiwaplus_consumer_app_user_session(start_date_str, end_date_str):
    connection = jiwa_db_conn()
    cursor = connection.cursor()
    
    query_1 = """
        SELECT 
            TO_CHAR(created_at + INTERVAL '7 hours', 'YYYY-MM-DD') as date,
            count(DISTINCT id) as total_session,
            COUNT(DISTINCT id_user) as total_users_open_jiwaplus
        FROM outlet_user_distances
        WHERE 
            TO_CHAR(created_at + INTERVAL '7 hours', 'YYYY-MM-DD') BETWEEN '{0}' AND '{1}
        GROUP by 1;
    """.format(start_date_str, end_date_str)
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def get_ksj_consumer_app_user_session(db_name, start_date_str, end_date_str):
    connection = ksj_link_db_conn(db_name)
    cursor = connection.cursor()
    
    query_1 = """
        SELECT 
            open_app.*,
            click_rider.total_click,
            click_rider.total_click_direction,
            click_rider.total_wa_direction,
            click_rider.total_click / open_app.total_rider_found as click_ratio_to_found_rider
        FROM
        (
            SELECT 
                date,
                SUM(total_users) as total_users_open_ksj,
                SUM(CASE WHEN type = 'Rider Found' THEN total_users ELSE 0 END) as total_rider_found,
                SUM(CASE WHEN type = 'Rider Not Found' THEN total_users ELSE 0 END) as total_rider_not_found,
                SUM(CASE WHEN type = 'Rider Found' THEN total_users ELSE 0 END) / SUM(total_users)  as rider_found_perc
            FROM
            (
            SELECT 
                TO_CHAR(created_at + INTERVAL '7 hours', 'YYYY-MM-DD') as date,
                'Rider Found' as type,
                COUNT(*) As total_session,
                COUNT(DISTINCT user_id) as total_users
            FROM ksj_rider_found_logs 
            WHERE 
                TO_CHAR(created_at + INTERVAL '7 hours', 'YYYY-MM-DD') BETWEEN '{0}' AND '{1}'
                AND riders_found_count >= 1
            GROUP BY 1
            
            UNION 
            
            SELECT 
                TO_CHAR(created_at + INTERVAL '7 hours', 'YYYY-MM-DD') as date,
                'Rider Not Found' as type,
                COUNT(*) As total_session,
                COUNT(DISTINCT user_id) as total_users
            FROM ksj_rider_found_logs
            WHERE 
                TO_CHAR(created_at + INTERVAL '7 hours', 'YYYY-MM-DD') BETWEEN '{0}' AND '{1}'
                AND riders_found_count= 0
            GROUP BY 1
            ) as open_screen
            GROUP BY 1
        ) AS open_app
        JOIN
        (
            SELECT 
                date,
                SUM(total_user) as total_click,
                SUM(CASE WHEN type = 'Click Direction' THEN total_user ELSE 0 END) as total_click_direction,
                SUM(CASE WHEN type = 'Click Whatsapp' THEN total_user ELSE 0 END) as total_wa_direction,
                SUM(CASE WHEN type = 'Click Whatsapp' THEN total_user ELSE 0 END) / SUM(total_user) as total_wa_perc
            FROM
            (
                SELECT 
                    TO_CHAR(timestamp, 'YYYY-MM-DD') as date,
                    'Click Direction' as type,
                    COUNT(DISTINCT event_id) as total_event,
                    COUNT(DISTINCT user_id) as total_user
                FROM ksj_user_click_tracker
                WHERE 
                    event_name = 'contactRider' 
                    AND event_label ='direction'
                    AND TO_CHAR(timestamp, 'YYYY-MM-DD') BETWEEN '{0}' AND '{1}'
                GROUP BY 1
                
                UNION 
                
                SELECT 
                    TO_CHAR(timestamp, 'YYYY-MM-DD') as date,
                    'Click Whatsapp' as type,
                    COUNT(DISTINCT event_id) as total_event,
                    COUNT(DISTINCT user_id) as total_user
                FROM ksj_user_click_tracker
                WHERE 
                    event_name = 'contactRider' 
                    AND event_label ='whatsapp'
                    AND TO_CHAR(timestamp, 'YYYY-MM-DD') BETWEEN '{0}' AND '{1}'
                GROUP BY 1
            ) as click_data
            GROUP BY 1
        ) AS click_rider ON open_app.date = click_rider.date
        ORDER BY 1 ASC;
    """.format(start_date_str, end_date_str)
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

## Get Date

In [ ]:
yesterday = datetime.today().date() - timedelta(days=1)

# start_date = today
# end_date = start_date - timedelta(days=1)

start_date_str = yesterday.strftime('%Y-%m-%d')
end_date_str = yesterday.strftime('%Y-%m-%d')

# start_date_str = '2026-05-01'
# end_date_str = '2026-05-21'

print("today     :", yesterday)
print("start_day :", start_date_str)
print("end_day   :", end_date_str)

# EXTRACT

In [ ]:
df_jiwaplus_session = get_jiwaplus_consumer_app_user_session(start_date_str, end_date_str)
df_ksj_session = get_ksj_consumer_app_user_session(config_ksj_link.d_jiwa_ksj, start_date_str, end_date_str)

print('jiwaplus session day: ', len(df_jiwaplus_session))
display(df_jiwaplus_session.head(3))

print('ksj session day: ', len(df_ksj_session))
display(df_ksj_session.head(3))

# TRANSFORM

In [ ]:
df_session_combined = df_jiwaplus_session.merge(df_ksj_session, on='date', how='outer')

df_session_combined.head(3)

# LOAD

## Engine

In [ ]:
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(
    hostsname=config_dl_dwh.hostname,
    user=config_dl_dwh.username,
    pw=config_dl_dwh.password,
    db=config_dl_dwh.db_target
))

## Dump to DWH

In [ ]:
df_session_combined.to_sql(
    name="daily_ksj_consumer_app_user_session",
    con=engine,
    if_exists="append",
    index=False,
    method="multi"
)
